# Install the Package
Here we're installing it directly from GitHub while it's in development.

In [2]:
!pip install 'vanna[flask,openai]'


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


# Download a Sample Database

In [3]:
import httpx

with open("Chinook.sqlite", "wb") as f:
    with httpx.stream("GET", "https://vanna.ai/Chinook.sqlite") as response:
        for chunk in response.iter_bytes():
            f.write(chunk)

# Imports

In [4]:
from vanna import Agent, AgentConfig
from vanna.servers.fastapi import VannaFastAPIServer
from vanna.core.registry import ToolRegistry
from vanna.core.user import UserResolver, User, RequestContext
from vanna.integrations.openai import OpenAILlmService
from vanna.tools import RunSqlTool, VisualizeDataTool
from vanna.integrations.sqlite import SqliteRunner
from vanna.tools.agent_memory import SaveQuestionToolArgsTool, SearchSavedCorrectToolUsesTool
from vanna.integrations.local.agent_memory import DemoAgentMemory
from vanna.capabilities.sql_runner import RunSqlToolArgs
from vanna.tools.visualize_data import VisualizeDataArgs

# Define your User Authentication
Here we're going to say that if you're logged in as `admin@example.com` then you're in the `admin` group, otherwise you're in the `user` group

In [5]:
class SimpleUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        # In production, validate cookies/JWTs here
        user_email = request_context.get_cookie('vanna_email')
        if not user_email:
            raise ValueError("Missing 'vanna_email' cookie for user identification")
        
        print(f"Resolving user for email: {user_email}")

        if user_email == "admin@example.com":
            return User(id="admin1", email=user_email, group_memberships=['admin'])
        
        return User(id="user1", email=user_email, group_memberships=['user'])

# Define the Tools and Access Control

In [6]:
tools = ToolRegistry()
tools.register_local_tool(RunSqlTool(sql_runner=SqliteRunner(database_path="./Chinook.sqlite")), access_groups=['admin', 'user'])
tools.register_local_tool(VisualizeDataTool(), access_groups=['admin', 'user'])
agent_memory = DemoAgentMemory(max_items=1000)
tools.register_local_tool(SaveQuestionToolArgsTool(), access_groups=['admin'])
tools.register_local_tool(SearchSavedCorrectToolUsesTool(), access_groups=['admin', 'user'])

In [8]:
import os
# Set up LLM
llm = OpenAILlmService(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

# Create agent with your options
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=SimpleUserResolver(),
    config=AgentConfig(),
    agent_memory=agent_memory
)

# 4. Create and run server
server = VannaFastAPIServer(agent)
server.run()

Your app is running at:
http://localhost:8000


INFO:     Started server process [30946]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51199 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:51199 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK
Resolving user for email: admin@example.com
INFO:     127.0.0.1:51199 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK
Resolving user for email: admin@example.com
INFO:     127.0.0.1:53305 - "POST /api/vanna/v2/chat_sse HTTP/1.1" 200 OK
Resolving user for email: admin@example.com


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [30946]


KeyboardInterrupt: 

In [15]:
from vanna.tools.agent_memory import SaveQuestionToolArgsTool
from vanna.core.tool import ToolContext
from vanna.core.user import User
from vanna.capabilities.sql_runner import RunSqlToolArgs

examples = [
    ("每个客户总消费Top 10", "SELECT CustomerId, SUM(Total) AS total_spent FROM invoices GROUP BY CustomerId ORDER BY total_spent DESC LIMIT 10"),
    ("每月销售额", "SELECT strftime('%Y-%m', InvoiceDate) AS month, SUM(Total) AS revenue FROM invoices GROUP BY month ORDER BY month"),
    ("曲库总曲目数", "SELECT COUNT(*) AS tracks FROM tracks")
]

save_tool = SaveQuestionToolArgsTool()
ctx = ToolContext(agent_memory=agent_memory, user=User(id="admin", email="admin@example.com", group_memberships=["admin"]))

for q, sql in examples:
    args = save_tool.get_args_schema()(question=q, tool_name="run_sql", args=RunSqlToolArgs(sql=sql))
    await save_tool.execute(ctx, args)

ValidationError: 2 validation errors for ToolContext
conversation_id
  Field required [type=missing, input_value={'agent_memory': <vanna.i..._memberships=['admin'])}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
request_id
  Field required [type=missing, input_value={'agent_memory': <vanna.i..._memberships=['admin'])}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [13]:
from vanna.core.user import RequestContext

ctx = RequestContext(cookies={"vanna_email": "admin@example.com"})

async def view_tool_memory():
    items = []
    async for comp in agent.send_message(
        request_context=ctx,
        message="请展示当前 Tool Memory 中保存的记录（包含问题、工具名、参数），最多展示100条"
    ):
        items.append(comp)
    return items

components = await view_tool_memory()
for c in components:
    print(c)

Resolving user for email: admin@example.com
timestamp='2025-11-27T07:25:59.417513' rich_component=StatusBarUpdateComponent(id='vanna-status-bar', type=<ComponentType.STATUS_BAR_UPDATE: 'status_bar_update'>, lifecycle=<ComponentLifecycle.CREATE: 'create'>, data={}, children=[], timestamp='2025-11-27T07:25:59.417464', visible=True, interactive=False, status='working', message='Processing your request...', detail='Analyzing query') simple_component=None
timestamp='2025-11-27T07:25:59.418949' rich_component=TaskTrackerUpdateComponent(id='vanna-task-tracker', type=<ComponentType.TASK_TRACKER_UPDATE: 'task_tracker_update'>, lifecycle=<ComponentLifecycle.CREATE: 'create'>, data={}, children=[], timestamp='2025-11-27T07:25:59.418923', visible=True, interactive=False, operation=<TaskOperation.ADD_TASK: 'add_task'>, task=Task(id='09e57fdc-769f-4839-b7bb-ff7c0ce4f25c', title='Load conversation context', description='Reading message history and user context', status='pending', progress=None, creat